# Harness Engineering Demo: Better Harness Beats Bigger Model

This notebook is built for a management-facing demo.

**Thesis:** a medium model with a good harness can beat a great model with a bad harness, because production quality depends on context, tools, validation, memory, safety gates, observability, and repair loops, not only raw model capability.

We will show the spectrum:

1. **Great model + bad harness**: bare prompt, no shared memory, no runbook enforcement.
2. **Medium model + hand-built harness**: explicit agents, shared memory, sensors, and scorecard.
3. **Medium model + SDK harness**: Strands-style abstraction for tools, hooks, memory, and multi-agent orchestration.
4. **Provider plug-and-play harness**: provider-specific harness lane, represented by DeepSeek adapter boundary.

The reliable part of the demo is deterministic. The live model section calls Ollama Cloud so management can see how this connects to real model backends.

## Demo Architecture

```text
Colab notebook
      ↓
Harness demo repo + Python SDKs
      ↓
Scenario: multi-agent incident response
      ↓
Scorecard: evidence, runbook, safety, memory, completeness
      ↓
Optional live calls to Ollama Cloud
```

Colab is the runtime. Ollama Cloud is the model backend.

## 1. Clone The Repo

Replace `REPO_URL` with your GitHub URL after pushing the project.

If you already uploaded this notebook into the cloned repo, skip this cell and `%cd` into the repo folder.

In [ ]:
# Replace this with your pushed GitHub repository URL.
REPO_URL = "https://github.com/YOUR_ORG/ollama-harness-engineering-demo.git"
REPO_DIR = "ollama-harness-engineering-demo"

from pathlib import Path
import os

# If we are not already inside the repo, clone it or move into an existing clone.
if not Path("pyproject.toml").exists():
    if Path(REPO_DIR).exists():
        os.chdir(REPO_DIR)
    else:
        if "YOUR_ORG" in REPO_URL:
            raise ValueError(
                "Replace REPO_URL with your GitHub repo URL, then rerun this cell. "
                "Example: https://github.com/my-org/ollama-harness-engineering-demo.git"
            )
        !git clone $REPO_URL
        os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
print("Project files:")
!ls -la

assert Path("requirements.txt").exists(), "requirements.txt not found. You are not inside the repo."
assert Path("pyproject.toml").exists(), "pyproject.toml not found. You are not inside the repo."

## 2. Install The Demo Dependencies

This happens inside Colab, so it does not depend on your office Mac allowing Python packages.

The important libraries are:

- `strands-agents`: SDK-level harness abstraction.
- `ollama`: direct calls to Ollama Cloud.
- `openai`: useful for OpenAI-compatible endpoints.
- `typer` and `rich`: CLI and readable scorecards.
- `pytest`: quick health check.

In [ ]:
from pathlib import Path

assert Path("requirements.txt").exists(), "Run the clone/%cd setup cell first. requirements.txt is missing here."
assert Path("pyproject.toml").exists(), "Run the clone/%cd setup cell first. pyproject.toml is missing here."

!pip install -r requirements.txt
!pip install -e .

## 3. Verify The Local Demo Works

This section does not call any live model. It proves the harness story reliably before we introduce network/model variability.

In [ ]:
!harness-demo list-lanes

In [ ]:
!harness-demo compare --scenario incident-response

### How To Narrate The Scorecard

- `raw-strong` is the cautionary baseline: a strong model with a bare prompt can sound plausible but fail production controls.
- `hand-built` shows what harness engineering does mechanically: agents, shared memory, runbook checks, safety checks, and scoring.
- `strands-sdk` shows that these controls are moving into reusable SDKs.
- `deepseek-provider` represents the plug-and-play provider-harness direction once provider-specific controls are packaged.

Management takeaway:

> The strategic asset is not one model. The strategic asset is the control layer around models.

## 4. Run Each Lane Individually

This makes the spectrum easier to explain live. Run one cell, pause, explain what changed, then run the next.

In [ ]:
!harness-demo run --scenario incident-response --lane raw-strong

In [ ]:
!harness-demo run --scenario incident-response --lane hand-built

In [ ]:
!harness-demo run --scenario incident-response --lane strands-sdk

In [ ]:
!harness-demo run --scenario incident-response --lane deepseek-provider

## 5. Optional: Run Tests

This is a quick confidence check before the management session.

In [ ]:
!python -m pytest -p no:cacheprovider

# Repo-Backed Multi-Agent Harness Walkthrough

From here on, the notebook is only the demo interface. The use case and harness logic come from the Python package in this repo.

We will import:

- `load_incident_scenario`: loads the incident ticket, logs, runbook, and prior memory.
- `Lane`: enumerates the harness spectrum.
- `RUNNERS`: executes each lane.
- reporting helpers: print the scorecard.

This is the important distinction: the demo is not a better prompt. It is a workflow with tools, shared memory, sensors, scoring, and lane-specific harness behavior.

In [ ]:
from pprint import pprint

from rich.console import Console

from harness_demo.domain import Lane
from harness_demo.reporting import print_comparison, print_result
from harness_demo.runners import RUNNERS
from harness_demo.scenarios import load_incident_scenario

console = Console(width=110)
scenario = load_incident_scenario("incident-response")

print("Loaded scenario:", scenario.id)
print("Scenario name:", scenario.name)

## 6. Show The Business Scenario

Before running any model lane, show management that this is a real workflow-shaped problem:

- incident ticket
- production logs
- approved runbook
- prior incident memory
- expected quality gates

A harness matters because the answer must coordinate all of these, not merely sound fluent.

In [ ]:
print("INCIDENT")
pprint(scenario.incident)

print("\nEXPECTED QUALITY GATES")
pprint(scenario.expected)

In [ ]:
print("LOGS")
print(scenario.logs)

print("RUNBOOK")
print(scenario.runbook)

print("PRIOR MEMORY")
print(scenario.prior_memory)

## 7. Run The Spectrum From Repo Code

This executes the existing Python runners in `src/harness_demo/runners/`.

The scorecard checks five production concerns:

- evidence was gathered from logs
- approved runbook steps were used
- unsafe actions were avoided
- prior memory was used
- final plan is complete

In [ ]:
results = [RUNNERS[lane](scenario) for lane in Lane]
print_comparison(console, results)

## 8. Inspect The Weak Harness Lane

`raw-strong` represents a strong model used as a bare assistant: no shared memory object, no runbook sensor, no reviewer gate, no repair loop.

This is the anti-pattern we want management to recognize.

In [ ]:
raw = RUNNERS[Lane.RAW_STRONG](scenario)
print_result(console, raw)

print("\nRaw lane memory object:")
pprint(raw.memory)

print("\nWhy it failed:")
for check, passed in raw.checks.items():
    if not passed:
        print("-", check)

## 9. Inspect The Hand-Built Harness Lane

This is the clearest harness engineering demonstration.

The lane coordinates specialist agents through a shared memory object:

- triage agent records incident facts
- log investigator extracts evidence
- runbook agent maps approved policy steps
- memory agent contributes prior lessons
- fix planner creates the remediation plan
- reviewer agent blocks unsafe actions

This is not prompt engineering. This is executable workflow control around model calls.

In [ ]:
hand_built = RUNNERS[Lane.HAND_BUILT](scenario)
print_result(console, hand_built)

print("\nIncident facts")
pprint(hand_built.memory.incident_facts)

print("\nEvidence gathered")
for item in hand_built.memory.evidence:
    print("-", item)

print("\nRunbook steps selected")
for item in hand_built.memory.runbook_steps:
    print("-", item)

print("\nPrior lessons used")
for item in hand_built.memory.prior_lessons:
    print("-", item)

print("\nReviewer objections")
print(hand_built.memory.reviewer_objections or "None")

print("\nFinal plan")
pprint(hand_built.memory.final_plan)

## 10. Inspect The Strands SDK Harness Lane

This lane represents the same production controls expressed through an SDK-level abstraction.

In the current repo, the lane is deterministic so the management demo is reliable. The point is architectural:

- hand-built lane shows the mechanics
- Strands lane shows how SDKs are packaging those mechanics: tools, hooks, shared context, session memory, observability, and multi-agent patterns

For a later iteration, this runner can be backed by real Strands `Agent`, `@tool`, hooks, and session managers while keeping the same scenario and scorer.

In [ ]:
strands_result = RUNNERS[Lane.STRANDS_SDK](scenario)
print_result(console, strands_result)

print("\nSame quality gates as hand-built lane:")
pprint(strands_result.checks)

print("\nSame shared-memory outcome shape:")
pprint(strands_result.memory.final_plan)

## 11. Inspect The Provider Harness Lane

This lane represents the plug-and-play direction: provider-specific harness behavior behind an adapter.

We should present this carefully:

- not as a verified DeepSeek package implementation yet
- as the target architecture for provider-specific protocol handling
- as the far end of the spectrum where more controls become packaged

In [ ]:
deepseek_result = RUNNERS[Lane.DEEPSEEK_PROVIDER](scenario)
print_result(console, deepseek_result)

print("\nAdapter boundary output shape:")
pprint(deepseek_result.memory.final_plan)

## 12. Side-By-Side Lane Internals

Use this cell when someone asks, "What changed between lanes?"

The answer should be: the harness maturity changed. The task stayed the same.

In [ ]:
for lane in Lane:
    result = RUNNERS[lane](scenario)
    print("\n" + "=" * 90)
    print(lane.value)
    print("score:", result.score)
    print("checks:")
    pprint(result.checks)
    print("takeaway:", result.business_takeaway)
    print("evidence count:", len(result.memory.evidence))
    print("runbook step count:", len(result.memory.runbook_steps))
    print("prior lesson count:", len(result.memory.prior_lessons))

# Optional Ollama Cloud Backend Check

Use this only after the repo-backed harness walkthrough.

The purpose is not to do a prompt-comparison demo. The purpose is to show that the same Colab environment can call Ollama Cloud, so the next implementation step can replace deterministic lane internals with live model calls.

In other words:

```text
Keep scenario, agents, memory, sensors, scorecard.
Swap deterministic agent internals for Ollama Cloud model calls.
```

## 13. Set Ollama Cloud API Key

Do not hardcode the key in the notebook. Use Colab Secrets if available, or enter it with `getpass`.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass("Enter OLLAMA_API_KEY: ")

print("OLLAMA_API_KEY configured:", bool(os.environ.get("OLLAMA_API_KEY")))

## 14. Verify Ollama Cloud Connectivity

This is intentionally a connectivity check, not the demo itself.

In [ ]:
from ollama import Client

ollama_client = Client(
    host="https://ollama.com",
    headers={"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]},
)

response = ollama_client.chat(
    model="gpt-oss:20b",
    messages=[{"role": "user", "content": "Reply with exactly: Ollama Cloud connected."}],
    stream=False,
)

print(response["message"]["content"])

## 15. Next Engineering Step: Live Agents Without Changing The Demo

The next repo change should add live model support inside the existing runners, for example:

```bash
harness-demo run --scenario incident-response --lane hand-built --model gpt-oss:20b --live
harness-demo run --scenario incident-response --lane raw-strong --model gpt-oss:120b --live
```

The important thing: the model calls should happen inside the existing harness workflow, not in standalone notebook prompt cells.

# Closing Narrative

What management should take away:

> Harness engineering is the control plane for AI applications.

The medium model is not magically smarter. It wins because the harness gives it:

- the right context
- controlled tools
- shared memory
- policy/runbook grounding
- reviewer checks
- objective sensors
- a repeatable scorecard

That is the difference between AI as a chat assistant and AI as a production application component.